# Sepsis Early-Warning Engine
**PhysioNet Sepsis Challenge 2019** — EDA and modeling notebook.  
Dvimidh Sule, BME — IIT Hyderabad.

## Load a single patient record

In [ ]:
%matplotlib inline
import pandas as pd

patient = pd.read_csv(
    'data/training_setA/training/p000001.psv',
    sep='|'
)

print(f'Shape: {patient.shape}  ({patient.shape[0]} hourly rows, {patient.shape[1]} columns)')
print()
print(patient.columns.tolist())

In [ ]:
patient.head()

The 41 columns split into four groups: 8 vitals measured roughly every hour, 26 lab values ordered sporadically (very sparse), 6 administrative fields, and the `SepsisLabel` target.

In [ ]:
VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']

LABS = [
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
    'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
    'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
    'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT',
    'WBC', 'Fibrinogen', 'Platelets'
]

ADMIN = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS']

LABEL = ['SepsisLabel']

total = len(VITALS) + len(LABS) + len(ADMIN) + len(LABEL)
print(f'Vitals: {len(VITALS)}, Labs: {len(LABS)}, Admin: {len(ADMIN)}, Label: {len(LABEL)}')
print(f'Total: {total} — matches columns: {total == patient.shape[1]}')

In [ ]:
# Missingness by group — labs are sparse by design; vitals are relatively dense
missing_pct = patient.isnull().mean().mul(100).round(1)
print('--- VITALS ---')
print(missing_pct[VITALS].to_string())
print()
print('--- LABS (first 6) ---')
print(missing_pct[LABS[:6]].to_string())
print('  ...')
print()
print('--- ADMIN ---')
print(missing_pct[ADMIN].to_string())

## Understanding the label

`SepsisLabel` is set to `1` starting **6 hours before** the patient first meets Sepsis-3 criteria, so a `1` at any given row means sepsis onset is at most 6 hours away. This gives the model a clinically meaningful prediction window rather than flagging patients only at the moment of diagnosis.

## Build the feature table

In [ ]:
raw = pd.read_csv('data/Dataset.csv')
raw = raw.rename(columns={'Patient_ID': 'patient_id'})
raw = raw.drop(columns=['Unnamed: 0', 'Hour'], errors='ignore')

print(f'Raw: {raw.shape}  |  Patients: {raw.patient_id.nunique()}  |  Prevalence: {raw.SepsisLabel.mean():.2%}')

In [ ]:
KEEP = ['patient_id'] + VITALS + LABEL
df = raw[KEEP].copy()

print(f'Feature table: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Sepsis prevalence: {df.SepsisLabel.mean():.2%}')
print(f'NaN stays in vitals — XGBoost handles natively, no imputation applied')

## Temporal feature engineering

A raw vital reading has no context. Adding a 3-hour rolling mean and 3-hour delta per vital gives the model trend information — rising HR over 3 hours is a very different signal from a stable HR at the same value. Both features map directly to FPGA-implementable primitives: a 3-deep shift register and a subtractor.

In [ ]:
grp = df.groupby("patient_id", sort=False)
for v in VITALS:
    df[f"{v}_mean3"] = grp[v].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f"{v}_mean6"] = grp[v].rolling(6, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f"{v}_delta3"] = grp[v].diff(3)
    df[f"{v}_missing"] = df[v].isna().astype("int8")

FEATURES_FULL = (VITALS + [f"{v}_mean3" for v in VITALS] + [f"{v}_mean6" for v in VITALS]
                 + [f"{v}_delta3" for v in VITALS] + [f"{v}_missing" for v in VITALS])
FEATURES_EDGE = ["Temp_mean3", "Resp_mean3", "HR_mean3", "SBP_mean3"]
FEATURES = FEATURES_FULL
print(f"FULL: {len(FEATURES_FULL)} feats | EDGE: {len(FEATURES_EDGE)} feats | df {df.shape}")

## Patient-level train/test split

Splitting at the patient level — not the row level — so no patient's data appears in both train and test. Row-level splitting would let the model memorise individual patient physiology and inflate ROC-AUC artificially.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['patient_id']))

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

train_patients = set(train_df['patient_id'].unique())
test_patients  = set(test_df['patient_id'].unique())

assert len(train_patients & test_patients) == 0, "Patient leakage detected"

print(f"Train: {train_df.shape}  |  {len(train_patients)} patients  |  sepsis prevalence: {train_df.SepsisLabel.mean():.2%}")
print(f"Test:  {test_df.shape}  |  {len(test_patients)} patients  |  sepsis prevalence: {test_df.SepsisLabel.mean():.2%}")
print(f"Leakage check passed — 0 patients in both splits")

In [ ]:
import xgboost as xgb

X_train = train_df[FEATURES].values
y_train = train_df['SepsisLabel'].values
X_test  = test_df[FEATURES].values
y_test  = test_df['SepsisLabel'].values

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'Negatives: {neg:,}  |  Positives: {pos:,}  |  scale_pos_weight: {scale_pos_weight:.1f}')

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)

params = {
    'objective':        'binary:logistic',
    'tree_method':      'hist',
    'device':           'cuda',
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric':      ['aucpr', 'auc'],
    'seed':             42,
}

booster = xgb.train(
    params, dtrain, num_boost_round=500,
    evals=[(dtrain, 'train'), (dtest, 'eval')],
    verbose_eval=50, early_stopping_rounds=20,
)

print(f'\nBest round: {booster.best_iteration}  |  Best eval-AUC: {booster.best_score:.5f}')

## Hyperparameter tuning (leak-free)

Optuna searches over 40 trials, maximising ROC-AUC on a **validation set carved from the training patients** (patient-level split). The test set is never seen during tuning — that would inflate the final score.

In [ ]:
import optuna
from sklearn.model_selection import GroupShuffleSplit

# Validation carved from TRAINING patients only - test set stays sealed
inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=7)
tr_idx, val_idx = next(inner.split(train_df, groups=train_df['patient_id']))
tr_sub, val_sub = train_df.iloc[tr_idx], train_df.iloc[val_idx]
assert len(set(tr_sub['patient_id']) & set(val_sub['patient_id'])) == 0

dtr  = xgb.DMatrix(tr_sub[FEATURES].values,  label=tr_sub['SepsisLabel'].values,  feature_names=FEATURES)
dval = xgb.DMatrix(val_sub[FEATURES].values, label=val_sub['SepsisLabel'].values, feature_names=FEATURES)
spw = (tr_sub['SepsisLabel'] == 0).sum() / (tr_sub['SepsisLabel'] == 1).sum()

def objective(trial):
    p = {'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cuda',
         'eval_metric': 'auc', 'scale_pos_weight': spw, 'seed': 42,
         'max_depth': trial.suggest_int('max_depth', 3, 10),
         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
         'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
         'gamma': trial.suggest_float('gamma', 0.0, 5.0),
         'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 10.0, log=True)}
    bst = xgb.train(p, dtr, num_boost_round=500, evals=[(dval, 'val')],
                    early_stopping_rounds=20, verbose_eval=False)
    return bst.best_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)
print(f'Best val ROC-AUC: {study.best_value:.4f}')

best_params = {**study.best_params, 'objective': 'binary:logistic', 'tree_method': 'hist',
               'device': 'cuda', 'eval_metric': ['aucpr', 'auc'], 'scale_pos_weight': spw, 'seed': 42}
booster = xgb.train(best_params, dtr, num_boost_round=500,
                    evals=[(dtr, 'train'), (dval, 'val')], early_stopping_rounds=20, verbose_eval=50)
print(f'Tuned booster - best round {booster.best_iteration}, val-AUC {booster.best_score:.4f}')

## Evaluation — ROC-AUC, PR-AUC, alarm fatigue

At 1.8% prevalence, ROC-AUC flatters — a model predicting 0 for everything still looks decent. PR-AUC is the honest metric: it only scores performance on the positive (sepsis) class. The confusion matrix is framed as alarm fatigue because in a clinical system, false positives have a real cost.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np

y_prob = booster.predict(dtest, iteration_range=(0, booster.best_iteration + 1))
y_pred = (y_prob >= 0.5).astype(int)

roc_auc     = roc_auc_score(y_test, y_prob)
pr_auc      = average_precision_score(y_test, y_prob)
baseline_pr = y_test.mean()

print(f'ROC-AUC : {roc_auc:.4f}')
print(f'PR-AUC  : {pr_auc:.4f}  (random baseline: {baseline_pr:.4f}, lift: {pr_auc/baseline_pr:.1f}x)')

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f'\nAt threshold 0.5:')
print(f'  True alarms  (TP): {tp:>6,}')
print(f'  False alarms (FP): {fp:>6,}  ← alarm fatigue')
print(f'  Missed sepsis(FN): {fn:>6,}  ← patient harm')
print(f'  Precision: {precision:.2%}  |  Recall: {recall:.2%}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax1.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.3f}')
ax1.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax1.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
ax1.legend()

prec, rec, _ = precision_recall_curve(y_test, y_prob)
ax2.plot(rec, prec, lw=2, label=f'AUC = {pr_auc:.3f}')
ax2.plot([0, 1], [baseline_pr, baseline_pr], color='k', linestyle='--', lw=1, label=f'Random ≈ {baseline_pr:.3f}')
ax2.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
ax2.legend()

plt.suptitle('Vitals-only XGBoost — held-out patients, no leakage', fontsize=12)
plt.tight_layout()
plt.savefig('figures/eval_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import json as _json
booster.save_model('models/sepsis_booster.json')
with open('models/sepsis_meta.json', 'w') as _f:
    _json.dump({'features': FEATURES, 'best_iteration': int(booster.best_iteration),
                'roc_auc': float(roc_auc), 'pr_auc': float(pr_auc), 'window': 3}, _f, indent=2)
print(f'Saved model + meta | ROC-AUC {roc_auc:.4f} | {len(FEATURES)} features')

## SHAP feature importance

SHAP picks *which* vitals drive the model. SIRS picks *thresholds* for the hardware. These are separate: SHAP is data-driven, SIRS is clinical definition.

In [ ]:
# SHAP via XGBoost-native TreeSHAP (pred_contribs) -- no external `shap` dependency
import numpy as np, matplotlib.pyplot as plt

rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=5000, replace=False)
X_sample = X_test[idx]

contribs = booster.predict(xgb.DMatrix(X_sample, feature_names=FEATURES), pred_contribs=True)
shap_values = np.asarray(contribs)[:, :-1]

mean_shap = np.abs(shap_values).mean(axis=0)
ranked = sorted(zip(FEATURES, mean_shap), key=lambda x: x[1], reverse=True)
print('Feature importance (mean |SHAP|):')
for feat, imp in ranked:
    print(f'  {feat:<15} {imp:.4f}')

order = np.argsort(mean_shap)[-20:]
fig, ax = plt.subplots(figsize=(10, 8))
for row, fi in enumerate(order):
    sv = shap_values[:, fi]; fv = X_sample[:, fi].astype(float)
    lo, hi = np.nanpercentile(fv, [5, 95]) if np.isfinite(fv).sum() > 1 else (0.0, 1.0)
    norm = np.clip((fv - lo) / (hi - lo + 1e-9), 0, 1); norm = np.where(np.isfinite(norm), norm, 0.5)
    y = row + (rng.random(len(sv)) - 0.5) * 0.6
    sc = ax.scatter(sv, y, c=norm, cmap='coolwarm', s=6, alpha=0.5, edgecolors='none')
ax.set_yticks(range(len(order))); ax.set_yticklabels([FEATURES[i] for i in order])
ax.axvline(0, color='gray', lw=0.8); ax.set_xlabel('SHAP value (impact on model output)')
ax.set_title('SHAP Summary (TreeSHAP)', fontsize=13, pad=12)
plt.tight_layout(); plt.savefig('figures/shap_summary.png', dpi=150, bbox_inches='tight'); plt.show()

## Edge model (FPGA candidate)

Server model maximises accuracy; this one maximises deployability. Uses only the 4 SHAP-top vitals as 3h rolling means, kept shallow (depth 4, ~60 trees) for synthesis. Scored on the same untouched test set so the accuracy gap is the honest price of hardware deployability.

In [ ]:
dtr_e   = xgb.DMatrix(tr_sub[FEATURES_EDGE].values,  label=tr_sub['SepsisLabel'].values,  feature_names=FEATURES_EDGE)
dval_e  = xgb.DMatrix(val_sub[FEATURES_EDGE].values, label=val_sub['SepsisLabel'].values, feature_names=FEATURES_EDGE)
dtest_e = xgb.DMatrix(test_df[FEATURES_EDGE].values, label=test_df['SepsisLabel'].values, feature_names=FEATURES_EDGE)

edge_params = {'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cuda',
               'eval_metric': ['aucpr', 'auc'], 'scale_pos_weight': spw, 'seed': 42,
               'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 1.0}
booster_edge = xgb.train(edge_params, dtr_e, num_boost_round=60,
                         evals=[(dval_e, 'val')], early_stopping_rounds=15, verbose_eval=20)

ye_prob  = booster_edge.predict(dtest_e, iteration_range=(0, booster_edge.best_iteration + 1))
edge_roc = roc_auc_score(y_test, ye_prob)
edge_pr  = average_precision_score(y_test, ye_prob)
print(f'SERVER: ROC-AUC {roc_auc:.4f} | PR-AUC {pr_auc:.4f}  ({len(FEATURES)} feats)')
print(f'EDGE:   ROC-AUC {edge_roc:.4f} | PR-AUC {edge_pr:.4f}  ({len(FEATURES_EDGE)} feats, depth4)')
print(f'Price of deployability: {roc_auc - edge_roc:+.4f} ROC-AUC')

booster_edge.save_model('models/sepsis_edge.json')

## Edge depth sensitivity

Depth costs hardware exponentially (`N*(2^d-1)` comparators). Sweep confirms whether depth 4 is the right lock for the chip.

In [ ]:
# Same edge features, same splits; only max_depth varies.
print(f'{"depth":>5} {"trees":>6} {"ROC-AUC":>8} {"PR-AUC":>7} {"~comparators":>13}')
edge_sweep = []
for d in [3, 4, 5, 6, 8, 10]:
    p = {'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cuda',
         'eval_metric': 'auc', 'scale_pos_weight': spw, 'seed': 42,
         'max_depth': d, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 1.0}
    b = xgb.train(p, dtr_e, num_boost_round=200,
                  evals=[(dval_e, 'val')], early_stopping_rounds=15, verbose_eval=False)
    pr = b.predict(dtest_e, iteration_range=(0, b.best_iteration + 1))
    roc = roc_auc_score(y_test, pr); prauc = average_precision_score(y_test, pr)
    ntrees = b.best_iteration + 1
    comps = ntrees * (2 ** d - 1)
    edge_sweep.append((d, ntrees, roc, prauc, comps))
    print(f'{d:>5} {ntrees:>6} {roc:>8.4f} {prauc:>7.4f} {comps:>13,}')

# Accuracy gained from depth 4 -> deepest
d4  = next(r for r in edge_sweep if r[0] == 4)
best = max(edge_sweep, key=lambda r: r[2])
print(f'\nDepth 4: ROC-AUC {d4[2]:.4f} at ~{d4[4]:,} comparators')
print(f'Best   : depth {best[0]}, ROC-AUC {best[2]:.4f} at ~{best[4]:,} comparators')
print(f'Going deeper buys {best[2]-d4[2]:+.4f} ROC-AUC for {best[4]/d4[4]:.0f}x the hardware.')

## Cross-validation (robustness)

5-fold `GroupKFold` on `patient_id` — every patient in exactly one test fold. Converts the single-split score into a mean +/- std across the whole cohort.

In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

X_all, y_all, groups = df[FEATURES_FULL].values, df['SepsisLabel'].values, df['patient_id'].values
n_rounds = booster.best_iteration + 1
cv_roc, cv_pr = [], []
for fold, (tr_i, te_i) in enumerate(GroupKFold(n_splits=5).split(X_all, y_all, groups), 1):
    assert len(set(groups[tr_i]) & set(groups[te_i])) == 0
    dtr_cv = xgb.DMatrix(X_all[tr_i], label=y_all[tr_i], feature_names=FEATURES_FULL)
    dte_cv = xgb.DMatrix(X_all[te_i], label=y_all[te_i], feature_names=FEATURES_FULL)
    spw_cv = (y_all[tr_i] == 0).sum() / (y_all[tr_i] == 1).sum()
    p = {**study.best_params, 'objective': 'binary:logistic', 'tree_method': 'hist',
         'device': 'cuda', 'eval_metric': 'auc', 'scale_pos_weight': spw_cv, 'seed': 42}
    prob = xgb.train(p, dtr_cv, num_boost_round=n_rounds).predict(dte_cv)
    cv_roc.append(roc_auc_score(y_all[te_i], prob)); cv_pr.append(average_precision_score(y_all[te_i], prob))
    print(f'Fold {fold}: ROC-AUC {cv_roc[-1]:.4f} | PR-AUC {cv_pr[-1]:.4f}')
print(f'5-fold ROC-AUC: {np.mean(cv_roc):.4f} +/- {np.std(cv_roc):.4f}')
print(f'5-fold PR-AUC:  {np.mean(cv_pr):.4f} +/- {np.std(cv_pr):.4f}')

## Operating point + calibration

Threshold chosen by clinical cost (false alarms per catch), not 0.5. `scale_pos_weight` distorts probabilities, so the reliability diagram shows over-prediction — rankings are good, displayed probabilities would need recalibration.

In [ ]:
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt

print(f'{"thr":>5} {"recall":>9} {"precision":>10} {"FA/catch":>9}')
for t in [0.30, 0.50, 0.70, 0.85, 0.95]:
    pred = y_prob >= t
    tp = int((pred & (y_test == 1)).sum()); fp = int((pred & (y_test == 0)).sum()); fn = int((~pred & (y_test == 1)).sum())
    recall = tp/(tp+fn) if (tp+fn) else 0; prec = tp/(tp+fp) if (tp+fp) else 0; fac = fp/tp if tp else float('inf')
    print(f'{t:5.2f} {recall:9.2%} {prec:10.2%} {fac:9.1f}')

frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy='quantile')
brier = brier_score_loss(y_test, y_prob)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
ax.plot(mean_pred, frac_pos, 'o-', lw=2, label=f'Server (Brier={brier:.4f})')
ax.set(xlabel='Mean predicted probability', ylabel='Observed sepsis fraction', title='Reliability diagram')
ax.legend(); plt.tight_layout(); plt.savefig('figures/calibration.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'Brier {brier:.4f} - miscalibration expected from scale_pos_weight; recalibrate for displayed probability.')

## Recalibration

Isotonic recalibration fit on the validation set (not test) fixes the probabilities distorted by `scale_pos_weight`. ROC-AUC is unchanged (monotonic map); only the Brier score / displayed probability improves. Calibrator saved for the  Bedside Monitor Dashboard demo.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt, joblib

p_val = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
iso = IsotonicRegression(out_of_bounds='clip').fit(p_val, val_sub['SepsisLabel'].values)
y_prob_cal = iso.predict(y_prob)

bb, ba = brier_score_loss(y_test, y_prob), brier_score_loss(y_test, y_prob_cal)
print(f'Brier {bb:.4f} -> {ba:.4f} | ROC-AUC {roc_auc_score(y_test, y_prob):.4f} -> {roc_auc_score(y_test, y_prob_cal):.4f} (unchanged)')

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
for probs, lab in [(y_prob, f'raw ({bb:.3f})'), (y_prob_cal, f'isotonic ({ba:.3f})')]:
    fp, mp = calibration_curve(y_test, probs, n_bins=10, strategy='quantile')
    ax.plot(mp, fp, 'o-', lw=2, label=lab)
ax.set(xlabel='Mean predicted prob', ylabel='Observed fraction', title='Calibration: raw vs isotonic'); ax.legend()
plt.tight_layout(); plt.savefig('figures/calibration_fixed.png', dpi=150, bbox_inches='tight'); plt.show()
joblib.dump(iso, 'models/sepsis_isotonic.joblib')

## Feature ceiling check

Adding 11 new features (rolling std, shock index HR/SBP, pulse pressure) moved 5-fold CV ROC-AUC by only +0.006 (0.718 -> 0.724) — within noise. The ~0.73 vitals-only ceiling is confirmed three ways: depth sweep, flat Optuna surface, and this feature ablation. Not folded in (marginal, and does not help the 4-vital edge model).